# MAMMAL Multi-Target Inference (protein + molecule)

Score (protein, molecule) pairs with a model fine-tuned by
[MAMMAL_multitarget_finetuning.ipynb](MAMMAL_multitarget_finetuning.ipynb).

Every row needs a protein sequence as well as a SMILES, and the prompt must be built the same way as during
fine-tuning, so `MAX_PROTEIN_TOKENS`, `MAX_SMILES_TOKENS` and `MAX_PROMPT_TOKENS` should match that run's
`config.json`.

To score molecules against a **new protein**, put its sequence in the protein column. Whether that works
depends on how many targets the model was trained on; with only a few it will not generalise.

**Environment:** run with the `biomed-mammal` kernel.

**Input:** a `.csv` or `.parquet` file with a protein column and a SMILES column. A 0/1 label column is
optional; if present, metrics are reported, overall and per target.

**Output**, in `<model run folder>/predictions/<PREDICTION_NAME>/`:
```
predictions.csv        # every input row, plus score, pred and rank (1 = highest, within each target)
metrics.json           # only if the file has labels
metrics_per_target.csv # only with labels and a target column
config.json            # settings used
```

## 1. Inputs and settings

In [ ]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/
DATASET = "mock-multitarget"

# Name of this prediction; outputs go to <model run folder>/predictions/<PREDICTION_NAME>/
PREDICTION_NAME = "test"

# Fine-tuned model: the best_model/ folder, or the run folder that contains it
MODEL_PATH = Path("../output") / DATASET / "multitarget" / "<RUN_NAME>_<timestamp>" / "best_model"

# Pairs to score
DATA_PATH = Path("../sample_data/mock_multitarget_test.csv")
PROTEIN_COL = "PROTEIN"
SMILES_COL = "SMILES"
LABEL_COL = "LABEL"     # set to None if the file has no labels
TARGET_COL = "TARGET"   # used for ranking and metrics per target; set to None if there is no such column
KEEP_COLS = []          # other input columns to copy into predictions.csv

In [ ]:
import os

BATCH_SIZE = 4
NUM_WORKERS = min(24, os.cpu_count())
THRESHOLD = 0.5
HIT_KS = [200, 100, 50]

# Keep these the same as the fine-tuning run (see its config.json)
MAX_PROTEIN_TOKENS = 1250
MAX_SMILES_TOKENS = 300
MAX_PROMPT_TOKENS = 1600

In [ ]:
import json

import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "mammal").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-alignment repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
MODEL_PATH = Path(MODEL_PATH).expanduser().resolve()
if not (MODEL_PATH / "config.json").exists() and (MODEL_PATH / "best_model").is_dir():
    MODEL_PATH = MODEL_PATH / "best_model"
assert MODEL_PATH.is_dir(), f"Model not found: {MODEL_PATH}"
assert Path(DATA_PATH).exists(), f"Data not found: {DATA_PATH}"
TOKENIZER_SOURCE = str(MODEL_PATH)

OUT_DIR = MODEL_PATH.parent / "predictions" / PREDICTION_NAME
assert not (OUT_DIR.exists() and any(OUT_DIR.iterdir())), (
    f"{OUT_DIR} already exists. Choose a new PREDICTION_NAME or delete that folder.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = dict(model_path=str(MODEL_PATH), data_path=str(DATA_PATH), protein_col=PROTEIN_COL,
              smiles_col=SMILES_COL, label_col=LABEL_COL, target_col=TARGET_COL,
              threshold=THRESHOLD, hit_ks=HIT_KS, max_protein_tokens=MAX_PROTEIN_TOKENS,
              max_smiles_tokens=MAX_SMILES_TOKENS, max_prompt_tokens=MAX_PROMPT_TOKENS)
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Model  : {MODEL_PATH}")
print(f"Data   : {DATA_PATH}")
print(f"Output : {OUT_DIR}")
print(f"Device : {DEVICE}")

## 2. Prepare data

Rows whose SMILES RDKit cannot read, or with no protein sequence, are kept in the output with an empty score.

In [ ]:
import numpy as np
import pandas as pd
import torch
from functools import partial
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from sklearn.metrics import average_precision_score, roc_auc_score

from fuse.data.utils.collates import CollateDefault
from mammal.keys import *
import pyarrow.parquet as pq
def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)
RDLogger.DisableLog("rdApp.*")

optional = [c for c in (LABEL_COL, TARGET_COL) if c]
input_df = read_table(DATA_PATH, [PROTEIN_COL, SMILES_COL, *KEEP_COLS], optional=optional)
input_df.insert(0, "row_index", range(len(input_df)))
HAS_LABELS = LABEL_COL is not None and LABEL_COL in input_df.columns and input_df[LABEL_COL].notna().any()
HAS_TARGETS = TARGET_COL is not None and TARGET_COL in input_df.columns

smiles = input_df[SMILES_COL].astype(str).str.strip()
protein = input_df[PROTEIN_COL].astype(str).str.strip()
valid = (input_df[SMILES_COL].notna() & (protein != "")
         & smiles.map(lambda s: Chem.MolFromSmiles(s) is not None))
pairs = pd.DataFrame({"protein": protein[valid], "smiles": smiles[valid]}).reset_index(drop=True)

print(f"Rows           : {len(input_df):,}")
print(f"Valid pairs    : {int(valid.sum()):,}")
if (~valid).any():
    print(f"Skipped        : {int((~valid).sum()):,} (kept with an empty score)")
print(f"Labels         : {'yes' if HAS_LABELS else 'no (metrics will be skipped)'}")
if HAS_TARGETS:
    print(f"Targets        : {input_df[TARGET_COL].nunique()}")

## 3. Load the model and build the prompts

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.model import Mammal

tokenizer_op = ModularTokenizerOp.from_pretrained(TOKENIZER_SOURCE)

NEG_TOKEN_ID = tokenizer_op.get_token_id("<0>")
POS_TOKEN_ID = tokenizer_op.get_token_id("<1>")
PAD_TOKEN_ID = tokenizer_op.get_token_id("<PAD>")
print(f"Tokenizer loaded. Answer tokens: <0> = {NEG_TOKEN_ID}, <1> = {POS_TOKEN_ID}")

In [ ]:
IGNORE_INDEX = -100
TOKEN_WEIGHTS_KEY = "data.token_weights"
MAX_ANSWER_TOKENS = 4
ANSWER_POSITION = 1  # the decoder writes <SENTINEL_ID_0> first, then the answer


def encoder_prompt(protein: str, smiles: str) -> str:
    """The text MAMMAL reads for one (target, molecule) pair.

    The tags say what each part is and how to split it: `AA` for the protein (one token per amino acid)
    and `SMILES` for the molecule. This follows the repository's own drug-target example
    (`mammal/examples/dti_bindingdb_kd`), with `<SENTINEL_ID_0>` marking where the answer belongs."""
    return (
        f"<@TOKENIZER-TYPE=AA><SENTINEL_ID_0>"
        f"<@TOKENIZER-TYPE=AA@MAX-LEN={MAX_PROTEIN_TOKENS}>"
        f"<MOLECULAR_ENTITY><MOLECULAR_ENTITY_GENERAL_PROTEIN>"
        f"<SEQUENCE_NATURAL_START>{protein}<SEQUENCE_NATURAL_END>"
        f"<@TOKENIZER-TYPE=SMILES@MAX-LEN={MAX_SMILES_TOKENS}>"
        f"<MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>"
        f"<SEQUENCE_NATURAL_START>{smiles}<SEQUENCE_NATURAL_END><EOS>"
    )


def prepare_sample(sample_dict: dict, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True) -> dict:
    """Tokenize one pair: the prompt, and (for training) the answer and its loss weight."""
    sample_dict[ENCODER_INPUTS_STR] = encoder_prompt(sample_dict["data.protein"], sample_dict["data.smiles"])
    tokenizer_op(
        sample_dict=sample_dict,
        key_in=ENCODER_INPUTS_STR,
        key_out_tokens_ids=ENCODER_INPUTS_TOKENS,
        key_out_attention_mask=ENCODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_PROMPT_TOKENS,
    )
    for k in (ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    if not with_label:
        return sample_dict

    label = int(sample_dict["data.label"])

    # What the model should write. The answer tokens are the same whichever tokenizer type opens the
    # string, so this matches the molecule-only notebooks exactly.
    sample_dict[LABELS_STR] = f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=LABELS_STR,
        key_out_tokens_ids=LABELS_TOKENS, key_out_attention_mask=LABELS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (LABELS_TOKENS, LABELS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])
    sample_dict[LABELS_TOKENS][sample_dict[LABELS_TOKENS] == PAD_TOKEN_ID] = IGNORE_INDEX

    sample_dict[DECODER_INPUTS_STR] = f"<@TOKENIZER-TYPE=SMILES><DECODER_START><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=DECODER_INPUTS_STR,
        key_out_tokens_ids=DECODER_INPUTS_TOKENS, key_out_attention_mask=DECODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (DECODER_INPUTS_TOKENS, DECODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    weight = pos_weight if label == 1 else 1.0
    sample_dict[TOKEN_WEIGHTS_KEY] = weight * (sample_dict[LABELS_TOKENS] != IGNORE_INDEX).float()
    return sample_dict


class PairDataset(Dataset):
    """One row per (target, molecule) pair, from a DataFrame with 'protein', 'smiles' and 'label'."""

    def __init__(self, df, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True):
        self.protein = df["protein"].tolist()
        self.smiles = df["smiles"].tolist()
        self.labels = df["label"].tolist() if with_label else [0] * len(df)
        self.tokenizer_op, self.pos_weight, self.with_label = tokenizer_op, pos_weight, with_label

    def __len__(self) -> int:
        return len(self.smiles)

    def __getitem__(self, idx: int) -> dict:
        sample = {"data.sample_id": idx, "data.protein": self.protein[idx],
                  "data.smiles": self.smiles[idx], "data.label": self.labels[idx]}
        return prepare_sample(sample, self.tokenizer_op, self.pos_weight, self.with_label)


def make_collate():
    """Trim the padding each batch does not need (10-20% faster)."""
    return CollateDefault(special_handlers_keys={
        ENCODER_INPUTS_TOKENS: partial(CollateDefault.crop_padding, pad_token_id=PAD_TOKEN_ID),
        ENCODER_INPUTS_ATTENTION_MASK: partial(CollateDefault.crop_padding, pad_token_id=False),
    })
def compute_metrics(labels, scores, hit_ks=None, threshold=0.5) -> dict:
    """hit@K, AUROC, PR AUC and the threshold metrics, as the MMELON notebooks report them."""
    hit_ks = HIT_KS if hit_ks is None else hit_ks
    labels, scores = np.asarray(labels).astype(int), np.asarray(scores, dtype=float)
    top = np.argsort(-scores, kind="stable")
    metrics = {f"hit@{k}": int(labels[top[:k]].sum()) for k in hit_ks}
    pred = scores > threshold
    tp, fp = int((pred & (labels == 1)).sum()), int((pred & (labels == 0)).sum())
    tn, fn = int((~pred & (labels == 0)).sum()), int((~pred & (labels == 1)).sum())
    both_classes = len(set(labels.tolist())) == 2
    metrics.update(
        rocauc=float(roc_auc_score(labels, scores)) if both_classes else float("nan"),
        pr_auc=float(average_precision_score(labels, scores)) if both_classes else float("nan"),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        specificity=tn / (tn + fp) if tn + fp else 0.0,
        accuracy=(tp + tn) / len(labels) if len(labels) else float("nan"),
    )
    return metrics
from tqdm.auto import tqdm

model = Mammal.from_pretrained(str(MODEL_PATH)).to(DEVICE).eval()


@torch.no_grad()
def score_pairs(df) -> np.ndarray:
    """The probability of `<1>` for each (protein, molecule) row, in order."""
    loader = DataLoader(PairDataset(df, tokenizer_op, with_label=False), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=make_collate(), num_workers=NUM_WORKERS)
    out = []
    for batch in tqdm(loader, desc="Scoring"):
        for k in (ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK):
            batch[k] = batch[k].to(DEVICE)
        result = model.generate(batch, output_scores=True, return_dict_in_generate=True, max_new_tokens=5)
        probs = result[SCORES][:, ANSWER_POSITION, :].float().cpu()
        pos, neg = probs[:, POS_TOKEN_ID], probs[:, NEG_TOKEN_ID]
        out.append((pos / (pos + neg + 1e-8)).numpy())
    return np.concatenate(out)


print(f"Loaded {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

## 4. Score the pairs

In [ ]:
%%time
scores = score_pairs(pairs)

predictions = input_df.copy()
predictions["score"] = np.nan
predictions.loc[valid, "score"] = scores
predictions["pred"] = (predictions["score"] > THRESHOLD).astype("Int64").where(valid)
# Rank within each target, since scores of different targets are not competing
if HAS_TARGETS:
    predictions["rank"] = (predictions.groupby(TARGET_COL)["score"]
                           .rank(ascending=False, method="first").astype("Int64"))
else:
    predictions["rank"] = predictions["score"].rank(ascending=False, method="first").astype("Int64")
predictions.to_csv(OUT_DIR / "predictions.csv", index=False)

print(f"\nSaved {len(predictions):,} rows to {OUT_DIR / 'predictions.csv'}")
display(predictions.drop(columns=[PROTEIN_COL]).sort_values("rank").head(10))

## 5. Metrics (if the file has labels)

In [ ]:
if not HAS_LABELS:
    print("No labels in the file; skipping metrics.")
else:
    scored = predictions[valid & predictions[LABEL_COL].notna()]
    metrics = compute_metrics(scored[LABEL_COL], scored["score"], hit_ks=HIT_KS, threshold=THRESHOLD)
    metrics = dict(n_pairs=len(scored), n_positives=int(scored[LABEL_COL].sum()), **metrics)
    (OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
    display(pd.Series(metrics, name="value").to_frame().round(4))

    if HAS_TARGETS:
        rows = []
        for target, g in scored.groupby(TARGET_COL):
            rows.append(dict(target=target, rows=len(g), positives=int(g[LABEL_COL].sum()),
                             **compute_metrics(g[LABEL_COL], g["score"], hit_ks=HIT_KS,
                                               threshold=THRESHOLD)))
        per_target = pd.DataFrame(rows)
        per_target.to_csv(OUT_DIR / "metrics_per_target.csv", index=False)
        display(per_target.round(4))

In [ ]:
# Files produced
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(OUT_DIR)):40s} {path.stat().st_size / 1e6:8.2f} MB")